<a href="https://colab.research.google.com/github/sadiyatamanna/EdvergenceX-Learning/blob/main/Context_Engineering_Lab_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Policy-Bound Decision System for Customer Support

The AI must:

    Answer only when context is sufficient

    Refuse when critical context is missing

    Ask clarification questions instead of guessing


## Why This Task Matters

In real systems:
- Context is often incomplete
- User data may be missing or conflicting
- AI must NOT guess or hallucinate

This task focuses on **controlled failure**, not better answers.

## Step 1: Add Guardrails to Static Context

We now enforce *when the AI should refuse to answer*.

In [1]:
SYSTEM_CONTEXT_GUARDED = """
You are a customer support assistant for an EdTech platform.

Rules:
- Do not assume missing information
- If required data is missing, ask a clarification question
- If policy cannot be applied, respond with "Unable to determine"
- Never hallucinate eligibility
- Be polite and professional
"""

## Step 2: Incomplete User Context (Simulated Real-World Issue)

Here, we intentionally remove critical information.

In [2]:
user_query = "Can I get a refund?"

user_profile_incomplete = {
    "role": "student"
    # Missing purchase date
    # Missing course progress
}

## Step 3: Assemble Context with Missing Information

Notice: we DO NOT fill missing values.

In [3]:
REFUND_POLICY = """
Refund Policy:
- Refunds are allowed only within 7 days of purchase
- Course progress must be below 20%
- Subscriptions are non-refundable
"""

In [4]:
final_prompt_incomplete = f"""
{SYSTEM_CONTEXT_GUARDED}

Company Policy:
{REFUND_POLICY}

User Profile:
- Role: {user_profile_incomplete['role']}

User Question:
{user_query}
"""

In [6]:
from google.colab import userdata
from openai import OpenAI

# 1. Load API key from Colab Secrets
MY_API_KEY = userdata.get("api_key")

# 2. Create OpenAI-compatible client
client = OpenAI(
    api_key=MY_API_KEY,
    base_url="https://nexusapi.navigatelabs.ai"
)

# 3. Send request to the model
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_CONTEXT_GUARDED
        },
        {
            "role": "user",
            "content": final_prompt_incomplete
        }
    ]
)

# 4. Print model response
print(response.choices[0].message.content)

Certainly! I can help you with practice questions.

To tailor them effectively, please tell me:
1.  What subject are you interested in?
2.  What is your current level for that subject (beginner, intermediate, or advanced)?


## What Should Happen?

Correct behavior:
- AI asks for missing information
OR
- AI says it cannot determine eligibility

This is **success**, not failure.

## Step 4: Conflicting Context

Real systems often receive contradictory data.

In [7]:
user_profile_conflict = {
    "role": "student",
    "course_progress": "10%",
    "purchase_days_ago": 30  # Conflicts with refund policy
}

In [8]:
final_prompt_conflict = f"""
{SYSTEM_CONTEXT_GUARDED}

Company Policy:
{REFUND_POLICY}

User Profile:
- Role: {user_profile_conflict['role']}
- Course Progress: {user_profile_conflict['course_progress']}
- Purchased: {user_profile_conflict['purchase_days_ago']} days ago

User Question:
{user_query}
"""

In [9]:
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "system", "content": SYSTEM_CONTEXT_GUARDED},
        {"role": "user", "content": final_prompt_conflict}
    ]
)

print(response.choices[0].message.content)

I apologize, but I am unable to determine if you can get practice questions based on the information provided in the Company Policy. The policy only details refund eligibility.


## Why This Is Important

The AI:
- Followed policy
- Did not hallucinate eligibility
- Handled conflicting data correctly

This is how **production AI systems** behave.

## Your Task (Hands-On)

Choose your domain and implement:
1. One missing-context scenario
2. One conflicting-context scenario
3. Guardrails that prevent guessing

You must show:
- AI refusal
- AI clarification
- AI safe fallback

# Domain:*COLLEGE COURSE ASSISTANCE*

# **Step 1:** Missing-Context Scenario
User Query:  
"Can I get practice questions?"

User Profile (Incomplete):

In [10]:
SYSTEM_CONTEXT_GUARDED = """
You are a college exam assistance AI.

Rules:
- Do not assume missing information
- If required data is missing, ask a clarification question
- If policy cannot be applied, respond with "Unable to determine"
- Never hallucinate eligibility or rules
- Be polite and professional
"""
user_query = "Can I get practice questions?"

user_profile_incomplete = {
    "role": "student"
    # Missing subject
    # Missing current level
}

EXAM_POLICY = """
Practice Policy:
- Students must specify the subject
- Questions are tailored to current level (beginner, intermediate, advanced)
"""

final_prompt_incomplete = f"""
{SYSTEM_CONTEXT_GUARDED}

College Policy:
{EXAM_POLICY}

User Profile:
- Role: {user_profile_incomplete['role']}

User Question:
{user_query}
"""

response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "system", "content": SYSTEM_CONTEXT_GUARDED},
        {"role": "user", "content": final_prompt_incomplete}
    ]
)

print(response.choices[0].message.content)


Hello! Yes, I can certainly help you with practice questions.

To tailor them effectively, please tell me:
1. What subject would you like practice questions for?
2. What is your current level in that subject (beginner, intermediate, or advanced)?


Policy:

Practice Policy:
- Students must specify the subject
- Questions are tailored to current level (beginner, intermediate, advanced)


AI Response:
Unable to determine what practice questions to provide.  
Could you please tell me:
1. Which subject you want practice questions for  
2. Your current level (beginner, intermediate, advanced)  

Once I have this, I can generate a plain test to check what you already know.


 Guardrail: No guessing, asks for missing context.

# **Step 2:** Conflicting-Context Scenario
User Query:  
"Can I get practice questions?"

User Profile (Conflict):

In [13]:
user_profile_conflict = {
    "role": "student",
    "subject": "Programming in Java",
    "level": "Intermediate",
    "completed_topics": ["Inheritance"]
}

final_prompt_conflict = f"""
{SYSTEM_CONTEXT_GUARDED}

College Policy:
{EXAM_POLICY}

User Profile:
- Role: {user_profile_conflict['role']}
- Subject: {user_profile_conflict['subject']}
- Level: {user_profile_conflict['level']}
- Completed Topics: {", ".join(user_profile_conflict['completed_topics'])}

User Question:
{user_query}
"""

response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "system", "content": SYSTEM_CONTEXT_GUARDED},
        {"role": "user", "content": final_prompt_conflict}
    ]
)

print(response.choices[0].message.content)


Certainly! Here is an intermediate-level practice question on Inheritance in Java:

**Question:**

Consider the following two Java classes:

```java
// Class A
public class Shape {
    private String color;

    public Shape(String color) {
        this.color = color;
    }

    public String getColor() {
        return color;
    }

    public void display() {
        System.out.println("This is a " + color + " shape.");
    }
}

// Class B
public class Circle extends Shape {
    private double radius;

    public Circle(String color, double radius) {
        super(color);
        this.radius = radius;
    }

    public double getRadius() {
        return radius;
    }

    // Question Part 1: Override the display method here.
    // The overridden method should print: "This is a [color] circle with radius [radius]."

    // Question Part 2: Add a new method named 'calculateArea' to the Circle class.
    // This method should return the area of the circle (π * radius * radius).
    //

AI Response (Safe Fallback):
Thank you for your request.  

You mentioned you are a beginner, but you have already completed Calculus, which is considered an advanced topic.  
Safe fallback: I will generate a plain test with **basic algebra and geometry** to confirm your foundation, and then suggest **development areas** like calculus practice if you are ready to progress.


Guardrail: AI detects conflict, resolves by safe fallback (foundation + development suggestion).

# **Step 3:** Output Structure (Diagnostic + Development)
When context is sufficient, the AI should produce:

Plain Test (What you already know):

Since the student is Intermediate but has already completed Inheritance (which is typically an advanced OOP concept), the AI should:

In [14]:
# Example of what the AI should generate when context is sufficient

plain_test = """
Plain Test (Java – Intermediate Level):
1. Write a Java class `Car` and extend it with a subclass `ElectricCar`.
2. Demonstrate method overriding with `toString()`.
3. Explain the difference between `super` and `this` keywords in Java.
"""

development_suggestions = """
Development Suggestions:
- Strengthen knowledge of polymorphism (dynamic method dispatch).
- Explore abstract classes vs interfaces in real-world design.
- Begin working with Java Collections Framework (ArrayList, HashMap).
- Progress toward exception handling and multithreading basics.
"""

print(plain_test)
print(development_suggestions)



Plain Test (Java – Intermediate Level):
1. Write a Java class `Car` and extend it with a subclass `ElectricCar`.
2. Demonstrate method overriding with `toString()`.
3. Explain the difference between `super` and `this` keywords in Java.


Development Suggestions:
- Strengthen knowledge of polymorphism (dynamic method dispatch).
- Explore abstract classes vs interfaces in real-world design.
- Begin working with Java Collections Framework (ArrayList, HashMap).
- Progress toward exception handling and multithreading basics.

